In [1]:
import pandas as pd
import json
import cv2
import numpy as np
from pathlib import Path
import os
from tqdm.notebook import tqdm

In [28]:
video_path = r"/home/share/schaer2/idtracking_keypoint/input/20200505155636_20200505174320_0_converted_small.mp4"
bbox = r"/home/share/schaer2/idtracking_keypoint/output/20200505155636_20200505174320_0_converted_small_bbox.json"
skpoint = r"/home/share/schaer2/idtracking_keypoint/output/results_20200505155636_20200505174320_0_converted_small.json"
timeline_behavior_path = r"/home/share/schaer2/idtracking_keypoint/output/oscillatory_periods_timeline.csv"
max_seconds = 60  # Set to -1 for full video duration


In [29]:
do_timeline = True if timeline_behavior_path else False

In [30]:
if do_timeline:
    # Load the CSV file using the correct path
    timeline_behavior_data = pd.read_csv(timeline_behavior_path)
    print("Timeline behavior data:")
    print(timeline_behavior_data.head())
    print(f"\nColumns: {list(timeline_behavior_data.columns)}")
    print(f"Shape: {timeline_behavior_data.shape}")
    timeline_behavior_data

else:
    timeline_behavior_data = None

Timeline behavior data:
                  signal_name  y_pos  start_time  duration          power  \
0  right_shoulder_angular_vel      0      535.20     12.36  179496.128800   
1  right_shoulder_angular_vel      0      552.92     16.32  377587.946748   
2  right_shoulder_angular_vel      0      651.36      6.36  320875.834919   
3  right_shoulder_angular_vel      0      723.76      8.76  238020.652526   
4  right_shoulder_angular_vel      0     1765.00      5.88  145485.384469   

   frequency            category  
0   0.554625  Angular Velocities  
1   0.513617  Angular Velocities  
2   0.513617  Angular Velocities  
3   0.323952  Angular Velocities  
4   0.949549  Angular Velocities  

Columns: ['signal_name', 'y_pos', 'start_time', 'duration', 'power', 'frequency', 'category']
Shape: (99, 7)


In [31]:
if do_timeline:
    # Show basic structure without the full data
    print("Basic data info:")
    print(f"Shape: {timeline_behavior_data.shape}")
    print(f"Columns: {list(timeline_behavior_data.columns)}")
    print("\nFirst few rows (limited columns):")
    if len(timeline_behavior_data.columns) > 5:
        print(timeline_behavior_data[timeline_behavior_data.columns[:5]].head())
    else:
        print(timeline_behavior_data.head())

    # Look for relevant columns for behaviors
    behavior_cols = [col for col in timeline_behavior_data.columns if any(word in col.lower() for word in ['behavior', 'start', 'end', 'left', 'right', 'shoulder'])]
    print(f"\nBehavior-related columns: {behavior_cols}")

    if behavior_cols:
        print("\nSample behavior data:")
        print(timeline_behavior_data[behavior_cols].head())

Basic data info:
Shape: (99, 7)
Columns: ['signal_name', 'y_pos', 'start_time', 'duration', 'power', 'frequency', 'category']

First few rows (limited columns):
                  signal_name  y_pos  start_time  duration          power
0  right_shoulder_angular_vel      0      535.20     12.36  179496.128800
1  right_shoulder_angular_vel      0      552.92     16.32  377587.946748
2  right_shoulder_angular_vel      0      651.36      6.36  320875.834919
3  right_shoulder_angular_vel      0      723.76      8.76  238020.652526
4  right_shoulder_angular_vel      0     1765.00      5.88  145485.384469

Behavior-related columns: ['start_time']

Sample behavior data:
   start_time
0      535.20
1      552.92
2      651.36
3      723.76
4     1765.00


In [32]:
if do_timeline:
    timeline_behavior_data['end_time'] = timeline_behavior_data['start_time'] + timeline_behavior_data['duration']
    # Check the modifier column for body parts
    print("Unique modifiers (body parts):")
    print(timeline_behavior_data['category'].unique())

    print("\nBehavior data grouped by modifier:")
    for modifier in timeline_behavior_data['category'].unique():
        data = timeline_behavior_data[timeline_behavior_data['category'] == modifier]
        print(f"\n{modifier}: {len(data)} events")
        print(data[['start_time', 'end_time']].head(3))

    timeline_behavior_data['start'] = timeline_behavior_data['start_time']
    timeline_behavior_data['end'] = timeline_behavior_data['end_time']
    timeline_behavior_data['modifier'] = timeline_behavior_data['category']

Unique modifiers (body parts):
['Angular Velocities' 'Joint Angles' 'Coordinates' 'Linear Velocities'
 'Distance Measures']

Behavior data grouped by modifier:

Angular Velocities: 14 events
   start_time  end_time
0      535.20    547.56
1      552.92    569.24
2      651.36    657.72

Joint Angles: 9 events
    start_time  end_time
14      535.72    547.92
15      553.20    569.28
16      534.88    546.80

Coordinates: 33 events
    start_time  end_time
23      533.76    547.76
24      552.64    565.28
25     1246.60   1262.24

Linear Velocities: 21 events
    start_time  end_time
56      186.80    199.60
57      535.16    548.28
58     1229.20   1239.84

Distance Measures: 22 events
    start_time  end_time
77      188.92    200.04
78      319.28    331.72
79      535.04    547.56


In [33]:
BEHAVIOR_COLORS

{'Angular Velocities': (255, 0, 0),
 'Joint Angles': (0, 255, 0),
 'Coordinates': (0, 0, 255),
 'Linear Velocities': (255, 255, 0),
 'Distance Measures': (255, 0, 255)}

In [35]:
# Enhanced implementation with timeline and behavior annotations

# OpenPose COCO connections for skeleton
COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

# Colors for 3 individuals (BGR format)
COLORS = [
    (255, 0, 0),    # Red
    (0, 255, 0),    # Green
    (0, 0, 255),    # Blue
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255),  # Yellow
    (128, 0, 0),    # Maroon
    (0, 128, 0),    # Dark Green
    (0, 0, 128),    # Navy
    (128, 128, 0),  # Olive
    (128, 0, 128),  # Purple
    (0, 128, 128),  # Teal
    (192, 192, 192)] # Silver]

# Behavior colors
BEHAVIOR_COLORS = {mod: COLORS[2:][i % len(COLORS[2:])] for i, mod in enumerate(timeline_behavior_data['modifier'].unique())}

def draw_skeleton(image, keypoints, connections, color, threshold=0.3):
    """Draw skeleton keypoints and connections"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if keypoints is None or len(keypoints) == 0:
        return img
    
    # Draw connections
    for connection in connections:
        idx1, idx2 = connection
        if (idx1 < len(keypoints) and idx2 < len(keypoints) and 
            keypoints[idx1][2] > threshold and keypoints[idx2][2] > threshold):
            x1, y1 = int(keypoints[idx1][0]), int(keypoints[idx1][1])
            x2, y2 = int(keypoints[idx2][0]), int(keypoints[idx2][1])
            cv2.line(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw keypoints
    for x, y, conf in keypoints:
        if conf > threshold:
            cv2.circle(img, (int(x), int(y)), 4, color, -1)
    
    return img

def draw_bbox(image, bbox, color, is_normalized=True):
    """Draw bounding box"""
    img = image.copy()
    h, w = img.shape[:2]
    
    if bbox is None:
        return img

    x1 = bbox[0]
    y1 = bbox[1]
    x2 = bbox[2]
    y2 = bbox[3]
    
    if is_normalized:
        # Convert normalized coordinates to pixel values
        x1 = int(x1 * w)
        y1 = int(y1 * h)
        x2 = int(x2 * w)
        y2 = int(y2 * h)
        
        # Ensure coordinates are within image bounds
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w - 1))
        y2 = max(0, min(y2, h - 1))

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    return img

def draw_timeline_with_legends(width, height, current_time, total_duration, behavior_data, timeline_height=60):
    """Draw timeline with behavior annotations, time cursor, and legends"""
    # Create extended timeline with space for legends
    legend_height = 80
    total_height = timeline_height + legend_height
    timeline = np.zeros((total_height, width, 3), dtype=np.uint8)
    
    # Draw timeline background (dark gray)
    cv2.rectangle(timeline, (0, 0), (width, timeline_height), (40, 40, 40), -1)
    
    # Draw behavior rectangles
    for _, row in behavior_data.iterrows():
        start_time = row['start']
        end_time = row['end']
        modifier = row['modifier']
        
        # Convert time to pixel positions
        start_x = int((start_time / total_duration) * width)
        end_x = int((end_time / total_duration) * width)
        
        # Get color for behavior type
        color = BEHAVIOR_COLORS.get(modifier, (128, 128, 128))  # Default gray
        
        # Draw behavior rectangle
        cv2.rectangle(timeline, (start_x, 10), (end_x, timeline_height-10), color, -1)
        
        # Add behavior label
        label = modifier.replace('Bras ', '')  # Shorten label
        text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)[0]
        text_x = start_x + 2
        text_y = timeline_height // 2 + text_size[1] // 2
        if text_x + text_size[0] < end_x:  # Only draw if fits
            cv2.putText(timeline, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Draw time cursor (white vertical line)
    cursor_x = int((current_time / total_duration) * width)
    cv2.line(timeline, (cursor_x, 0), (cursor_x, timeline_height), (255, 255, 255), 2)
    
    # Draw time markers
    for i in range(0, int(total_duration), max(1, int(total_duration // 10))):
        marker_x = int((i / total_duration) * width)
        cv2.line(timeline, (marker_x, timeline_height-5), (marker_x, timeline_height), (200, 200, 200), 1)
        # Add time label
        time_label = f"{i//60}:{i%60:02d}" if i >= 60 else f"{i}s"
        cv2.putText(timeline, time_label, (marker_x+2, timeline_height-15), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)
    
    # Draw legends below timeline
    legend_y_start = timeline_height + 10
    
    # Individual colors legend (left side)
    legend_x = 20
    cv2.putText(timeline, "Individuals:", (legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    individual_labels = ["Child", "Clinician", "Parent"]
    for i, (color, label) in enumerate(zip(COLORS, individual_labels)):
        y_pos = legend_y_start + 20 + (i * 18)
        # Draw color rectangle
        cv2.rectangle(timeline, (legend_x, y_pos-8), (legend_x+15, y_pos+2), color, -1)
        # Draw label
        cv2.putText(timeline, label, (legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Behavior colors legend (right side)
    behavior_legend_x = width // 2 + 50
    cv2.putText(timeline, "Repetitive Behaviors:", (behavior_legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    behavior_items = [
        ("Bras Gauche", "Left Arm", BEHAVIOR_COLORS.get('Bras Gauche', (128, 128, 128))),
        ("Bras Droit", "Right Arm", BEHAVIOR_COLORS.get('Bras Droit', (128, 128, 128))),
        ("Shoulder", "Shoulder", BEHAVIOR_COLORS.get('Shoulder', (128, 128, 128)))
    ]
    
    for i, (label, color) in enumerate(BEHAVIOR_COLORS.items()):
        y_pos = legend_y_start + 20 + (i * 18)
        # Draw color rectangle
        cv2.rectangle(timeline, (behavior_legend_x, y_pos-8), (behavior_legend_x+15, y_pos+2), color, -1)
        # Draw label
        cv2.putText(timeline, label, (behavior_legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Add current time display (top right)
    time_text = f"Time: {int(current_time//60)}:{int(current_time%60):02d}"
    time_size = cv2.getTextSize(time_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    cv2.putText(timeline, time_text, (width - time_size[0] - 10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return timeline

def extract_frame_data(bbox_data, keypoint_data, frame_idx):
    """Extract bbox and keypoints for all individuals in a frame"""
    # Extract bboxes
    bboxes = {}
    if str(frame_idx) in bbox_data:
        frame_bboxes = bbox_data[str(frame_idx)]
        bboxes = frame_bboxes
    
    # Extract keypoints
    keypoints_dict = {}
    if 'instance_info' in keypoint_data:
        # Find frame data
        frame_data = None
        for data in keypoint_data['instance_info']:
            if data['frame_id'] == frame_idx:
                frame_data = data
                break
        
        if frame_data and 'instances' in frame_data:
            instances = frame_data['instances']
            for i, person in enumerate(instances):
                if person is not None and 'keypoints' in person and 'keypoint_scores' in person and 'keypoints_label' in person:
                    kp = np.array(person['keypoints'])  # (17, 2)
                    scores = np.array(person['keypoint_scores'])  # (17,)
                    identity = str(int(person['keypoints_label']))  # (17,)
                    # Combine to (17, 3) format
                    combined = np.zeros((17, 3))
                    combined[:, :2] = kp
                    combined[:, 2] = scores
                    keypoints_dict[int(identity)] = combined
                else:
                    tqdm.write(f"Warning: Frame {frame_idx} individual {i} missing keypoints data.")
    
    return bboxes, keypoints_dict

def create_enhanced_video(video_path, bbox_data, keypoint_data, behavior_data, output_path, max_seconds=10):
    """Create enhanced video with timeline and behavior annotations"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    if max_seconds <= 0:
        print("Processing full video duration...")
        max_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_duration = max_frames / fps
    else:
        max_frames = int(fps * max_seconds)
        total_duration = max_seconds
    
    print(f"Processing {max_frames} frames ({total_duration:.1f}s) at {fps} fps")
    
    # Timeline height (increased for legends)
    if do_timeline:
        timeline_height = 140  # 60 for timeline + 80 for legends
    else:
        timeline_height = 0
    
    # Output video dimensions: width*2 for 2x2 grid, height*2 + timeline_height
    output_width = width * 2
    output_height = height * 2 + timeline_height
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (output_width, output_height))
    
    for frame_idx in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Current time in seconds
        current_time = frame_idx / fps
        
        # Get data for all individuals
        bboxes, keypoints_dict = extract_frame_data(bbox_data, keypoint_data, frame_idx)
        
        # Create 4 quadrants
        top_left = frame.copy()  # Original
        
        top_right = frame.copy()  # With bboxes
        for idx, (identity, bbox) in enumerate(bboxes.items()):
            if bbox is not None:
                # Use COLORS cycling if more individuals than colors
                color = COLORS[idx % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
                top_right = draw_bbox(top_right, bbox, color)
        
        bottom_left = np.zeros_like(frame)  # Keypoints on black
        for idx, (identity, keypoints) in enumerate(keypoints_dict.items()):
            if keypoints is not None:
                color = COLORS[idx % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
                bottom_left = draw_skeleton(bottom_left, keypoints, COCO_CONNECTIONS, color)
        
        bottom_right = frame.copy()  # Keypoints on video
        for idx, (identity, keypoints) in enumerate(keypoints_dict.items()):
            if keypoints is not None:
                color = COLORS[idx % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
                bottom_right = draw_skeleton(bottom_right, keypoints, COCO_CONNECTIONS, color)
        
        # Combine into 2x2 grid
        top_row = np.hstack((top_left, top_right))
        bottom_row = np.hstack((bottom_left, bottom_right))
        video_grid = np.vstack((top_row, bottom_row))
        
        # Create timeline with legends
        if do_timeline:
            timeline = draw_timeline_with_legends(output_width, timeline_height, current_time, total_duration, behavior_data)
            # Combine video grid with timeline (timeline in the middle)
            final_frame = np.vstack((video_grid[:height*2//2], timeline, video_grid[height*2//2:]))
        else:
            # Combine video grid without timeline
            final_frame = np.vstack((video_grid[:height*2//2], video_grid[height*2//2:]))
        
        out.write(final_frame)
        
        if frame_idx % 50 == 0:
            print(f"Processed {frame_idx}/{max_frames} frames")
    
    cap.release()
    out.release()
    print(f"✅ Enhanced video saved: {output_path}")

# Load data and create enhanced test video
print("Loading data...")
with open(bbox, 'r') as f:
    bbox_data = json.load(f)
with open(skpoint, 'r') as f:
    keypoint_data = json.load(f)

max_seconds = -1
# Create output path
if max_seconds <= 0:
    output_path = str(Path(skpoint).parent / f"{os.path.basename(video_path).split('.')[0] }_compiled_video_full.mp4")
else:
    output_path = str(Path(skpoint).parent / f"{os.path.basename(video_path).split('.')[0] }_compiled_video_{max_seconds}s.mp4")

print("Creating 10-second enhanced video with timeline...")
create_enhanced_video(video_path, bbox_data, keypoint_data, timeline_behavior_data, output_path, max_seconds=max_seconds)

Loading data...
Creating 10-second enhanced video with timeline...
Processing full video duration...
Processing 45000 frames (1800.0s) at 25.0 fps
Processed 0/45000 frames
Processed 50/45000 frames
Processed 100/45000 frames
Processed 150/45000 frames
Processed 200/45000 frames
Processed 250/45000 frames
Processed 300/45000 frames
Processed 350/45000 frames
Processed 400/45000 frames
Processed 450/45000 frames
Processed 500/45000 frames
Processed 550/45000 frames
Processed 600/45000 frames
Processed 650/45000 frames
Processed 700/45000 frames
Processed 750/45000 frames
Processed 800/45000 frames
Processed 850/45000 frames
Processed 900/45000 frames
Processed 950/45000 frames
Processed 1000/45000 frames
Processed 1050/45000 frames
Processed 1100/45000 frames
Processed 1150/45000 frames
Processed 1200/45000 frames
Processed 1250/45000 frames
Processed 1300/45000 frames
Processed 1350/45000 frames
Processed 1400/45000 frames
Processed 1450/45000 frames
Processed 1500/45000 frames
Processe